In [ ]:
# ==================================================================================
# RoseSSL-CBAMNet — ROBUSTNESS, GENERALIZATION & EMBEDDING-QUALITY ANALYSIS
#
#   PART D: Noise/corruption robustness   (Gaussian noise, blur, brightness, JPEG)
#   PART E: Adversarial robustness        (FGSM; optional PGD)
#   PART F: Cross-domain / cross-season generalization test (plug-in folder)
#   PART G: Linear probing accuracy       (frozen encoder + single linear layer)
#   PART H: Embedding-quality metrics     (Silhouette score, Davies-Bouldin index)
#
# PREREQUISITES (from your existing pipeline):
#   - ssl_encoder.keras         -> saved by your Part-1 SimCLR script
#   - a trained FULL classifier -> saved by your Part-2 script or the advanced-
#                                  analysis script (SimCLR + CBAM + head)
#   Update SSL_ENCODER_PATH / FULL_MODEL_PATH below to point at your actual files.
#
# This script is self-contained: it re-declares the CBAM block, data pipeline, and
# a fixed held-out test split (same FIXED_TEST_SEED convention as the advanced
# analysis script) so it can be run independently.
# ==================================================================================

import os
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import layers, Model

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, silhouette_score,
    davies_bouldin_score
)
from sklearn.manifold import TSNE

try:
    import cv2
except ImportError:
    raise ImportError("This script needs OpenCV. Run: pip install opencv-python --break-system-packages")

print("TensorFlow:", tf.__version__)

# ========================== CONFIG ==========================
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 4
COMBINED_PATH = "/kaggle/input/datasets/sabbir4724/rose-leaf"
SSL_ENCODER_PATH = "ssl_encoder.keras"                 # from Part-1 SimCLR script
FULL_MODEL_PATH = "AgriSSL_CBAMNet_labdata.keras"       # your trained SimCLR+CBAM classifier
FIXED_TEST_SEED = 42
OUTPUT_DIR = "robustness_embedding_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Optional: point this at a folder of NEW images (different season/lighting/camera),
# organized the same way as COMBINED_PATH (one sub-folder per class). Leave as None
# to skip PART F.
CROSS_DOMAIN_PATH = None   # e.g. "/kaggle/working/rose_new_season"

RUN_PGD = False   # PGD is more expensive (multi-step); flip to True if you want it

AUTOTUNE = tf.data.AUTOTUNE

# ========================== FILE LIST + FIXED TEST SPLIT (same convention as before) ==========================
class_names = sorted(os.listdir(COMBINED_PATH))
class_to_idx = {cls: i for i, cls in enumerate(class_names)}

image_paths, labels = [], []
for cls in class_names:
    folder = os.path.join(COMBINED_PATH, cls)
    for img in os.listdir(folder):
        image_paths.append(os.path.join(folder, img))
        labels.append(class_to_idx[cls])
image_paths = np.array(image_paths)
labels = np.array(labels)

trainpool_paths, test_paths, trainpool_labels, test_labels = train_test_split(
    image_paths, labels, test_size=0.20, random_state=FIXED_TEST_SEED, stratify=labels
)
print(f"Held-out TEST set (fixed, seed={FIXED_TEST_SEED}): {len(test_paths)} images")

# ========================== LOAD TEST SET INTO MEMORY (needed for pixel-level corruption/attack) ==========================
def load_image_array(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    return img.numpy().astype(np.float32)   # kept in 0-255 range; model applies preprocess_input internally

print("Loading held-out test images into memory for corruption/attack experiments...")
X_test = np.stack([load_image_array(p) for p in test_paths]).astype(np.float32)
y_test = test_labels.copy()
print("X_test shape:", X_test.shape)

def predict_in_batches(model, images_array, batch_size=BATCH_SIZE):
    probs = []
    for start in range(0, len(images_array), batch_size):
        batch = images_array[start:start + batch_size]
        probs.append(model.predict(batch, verbose=0))
    return np.concatenate(probs, axis=0)

# ========================== CBAM (unchanged) ==========================
class ChannelMean(layers.Layer):
    def call(self, x):
        return tf.reduce_mean(x, axis=-1, keepdims=True)

class ChannelMax(layers.Layer):
    def call(self, x):
        return tf.reduce_max(x, axis=-1, keepdims=True)

def cbam_block(input_feature, ratio=8):
    channel = input_feature.shape[-1]
    avg_pool = layers.Reshape((1, 1, channel))(layers.GlobalAveragePooling2D()(input_feature))
    max_pool = layers.Reshape((1, 1, channel))(layers.GlobalMaxPooling2D()(input_feature))
    shared_dense_1 = layers.Dense(channel // ratio, activation="relu")
    shared_dense_2 = layers.Dense(channel)
    avg_out = shared_dense_2(shared_dense_1(avg_pool))
    max_out = shared_dense_2(shared_dense_1(max_pool))
    channel_attention = layers.Activation("sigmoid")(layers.Add()([avg_out, max_out]))
    channel_refined = layers.Multiply()([input_feature, channel_attention])
    concat = layers.Concatenate(axis=-1)([ChannelMean()(channel_refined), ChannelMax()(channel_refined)])
    spatial_attention = layers.Conv2D(1, 7, padding="same", activation="sigmoid")(concat)
    return layers.Multiply()([channel_refined, spatial_attention])

# ========================== LOAD THE TRAINED FULL MODEL ==========================
if not os.path.exists(FULL_MODEL_PATH):
    raise FileNotFoundError(
        f"Could not find '{FULL_MODEL_PATH}'. Point FULL_MODEL_PATH at your trained "
        f"SimCLR+CBAM classifier (.keras file) before running this script."
    )
full_model = tf.keras.models.load_model(
    FULL_MODEL_PATH, custom_objects={"ChannelMean": ChannelMean, "ChannelMax": ChannelMax}
)
print("Loaded full trained model:", FULL_MODEL_PATH)

# Clean baseline (severity 0) — everything else is measured as a drop from this
y_prob_clean = predict_in_batches(full_model, X_test)
y_pred_clean = np.argmax(y_prob_clean, axis=1)
acc_clean = accuracy_score(y_test, y_pred_clean)
_, _, f1_clean, _ = precision_recall_fscore_support(y_test, y_pred_clean, average="weighted", zero_division=0)
print(f"Clean test accuracy: {acc_clean:.4f}  |  Clean weighted F1: {f1_clean:.4f}")

# ==================================================================================
# PART D: NOISE / CORRUPTION ROBUSTNESS
# ==================================================================================
print("\n" + "=" * 78)
print("PART D: ROBUSTNESS UNDER IMAGE PERTURBATIONS")
print("=" * 78)

def apply_gaussian_noise(images, sigma):
    if sigma == 0:
        return images
    noise = np.random.normal(0, sigma, images.shape).astype(np.float32)
    return np.clip(images + noise, 0, 255)

def apply_gaussian_blur(images, ksize):
    if ksize <= 1:
        return images
    out = np.empty_like(images)
    for i in range(images.shape[0]):
        out[i] = cv2.GaussianBlur(images[i], (ksize, ksize), 0)
    return out

def apply_brightness(images, factor):
    if factor == 1.0:
        return images
    return np.clip(images * factor, 0, 255)

def apply_jpeg_compression(images, quality):
    if quality >= 100:
        return images
    out = np.empty_like(images)
    params = [int(cv2.IMWRITE_JPEG_QUALITY), quality]
    for i in range(images.shape[0]):
        bgr = cv2.cvtColor(images[i].astype(np.uint8), cv2.COLOR_RGB2BGR)
        _, enc = cv2.imencode(".jpg", bgr, params)
        dec_bgr = cv2.imdecode(enc, cv2.IMREAD_COLOR)
        out[i] = cv2.cvtColor(dec_bgr, cv2.COLOR_BGR2RGB).astype(np.float32)
    return out

CORRUPTIONS = {
    "Gaussian Noise (sigma)": (apply_gaussian_noise, [0, 10, 20, 30]),
    "Gaussian Blur (kernel)": (apply_gaussian_blur, [1, 3, 5, 7]),
    "Brightness (factor)":    (apply_brightness, [1.0, 1.3, 0.7, 0.5]),
    "JPEG Compression (quality)": (apply_jpeg_compression, [100, 50, 30, 10]),
}

robustness_rows = []
for corruption_name, (fn, severities) in CORRUPTIONS.items():
    for sev in severities:
        X_corrupt = fn(X_test.copy(), sev)
        y_prob = predict_in_batches(full_model, X_corrupt)
        y_pred = np.argmax(y_prob, axis=1)
        acc = accuracy_score(y_test, y_pred)
        _, _, f1, _ = precision_recall_fscore_support(y_test, y_pred, average="weighted", zero_division=0)
        robustness_rows.append({
            "Corruption": corruption_name, "Severity": sev,
            "Accuracy": acc, "F1": f1,
            "Accuracy Drop (pp)": (acc_clean - acc) * 100
        })
        print(f"{corruption_name:28s} severity={sev!s:>6} -> Acc={acc:.4f}  (drop={100*(acc_clean-acc):.2f} pp)")

robust_df = pd.DataFrame(robustness_rows)
robust_df.to_csv(os.path.join(OUTPUT_DIR, "robustness_perturbation_results.csv"), index=False)

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
axes = axes.flatten()
for i, (corruption_name, (_, severities)) in enumerate(CORRUPTIONS.items()):
    sub = robust_df[robust_df["Corruption"] == corruption_name]
    axes[i].plot(range(len(severities)), sub["Accuracy"], marker="o", color="#4C72B0")
    axes[i].axhline(acc_clean, color="gray", linestyle="--", label="Clean accuracy")
    axes[i].set_xticks(range(len(severities)))
    axes[i].set_xticklabels(severities)
    axes[i].set_title(corruption_name)
    axes[i].set_xlabel("Severity")
    axes[i].set_ylabel("Accuracy")
    axes[i].legend(fontsize=8)
plt.suptitle("Robustness Analysis under Image Perturbations")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "robustness_perturbation_curves.png"), dpi=600)
plt.show()

print("\nRobustness summary table (paste into paper as 'Table: Robustness under Image Perturbations'):")
print(robust_df.to_string(index=False))

# ==================================================================================
# PART E: ADVERSARIAL ROBUSTNESS (FGSM; optional PGD)
# ==================================================================================
print("\n" + "=" * 78)
print("PART E: ADVERSARIAL ROBUSTNESS (FGSM" + (" + PGD)" if RUN_PGD else ", optional)"))
print("=" * 78)

def fgsm_batch(model, images, labels_int, epsilon):
    images_t = tf.convert_to_tensor(images, dtype=tf.float32)
    labels_t = tf.convert_to_tensor(labels_int, dtype=tf.int64)
    with tf.GradientTape() as tape:
        tape.watch(images_t)
        preds = model(images_t, training=False)
        loss = tf.keras.losses.sparse_categorical_crossentropy(labels_t, preds)
    grad = tape.gradient(loss, images_t)
    adv = images_t + epsilon * tf.sign(grad)
    adv = tf.clip_by_value(adv, 0.0, 255.0)
    return adv.numpy()

def fgsm_attack_in_batches(model, images, labels_int, epsilon, batch_size=BATCH_SIZE):
    out = np.empty_like(images)
    for start in range(0, len(images), batch_size):
        end = start + batch_size
        out[start:end] = fgsm_batch(model, images[start:end], labels_int[start:end], epsilon)
    return out

EPSILONS = [0, 2, 4, 8, 16]  # perturbation budget in raw pixel units (0-255 scale)

adv_rows = []
for eps in EPSILONS:
    if eps == 0:
        X_adv = X_test
    else:
        X_adv = fgsm_attack_in_batches(full_model, X_test, y_test, eps)
    y_prob = predict_in_batches(full_model, X_adv)
    y_pred = np.argmax(y_prob, axis=1)
    acc = accuracy_score(y_test, y_pred)
    adv_rows.append({"Attack": "FGSM", "Epsilon": eps, "Accuracy": acc,
                      "Accuracy Drop (pp)": (acc_clean - acc) * 100})
    print(f"FGSM epsilon={eps:>3d}/255 -> Acc={acc:.4f}  (drop={100*(acc_clean-acc):.2f} pp)")

if RUN_PGD:
    def pgd_attack_in_batches(model, images, labels_int, epsilon, alpha=2.0, iters=10, batch_size=BATCH_SIZE):
        out = np.empty_like(images)
        for start in range(0, len(images), batch_size):
            end = start + batch_size
            orig = tf.convert_to_tensor(images[start:end], dtype=tf.float32)
            lbl = tf.convert_to_tensor(labels_int[start:end], dtype=tf.int64)
            adv = tf.identity(orig)
            for _ in range(iters):
                with tf.GradientTape() as tape:
                    tape.watch(adv)
                    preds = model(adv, training=False)
                    loss = tf.keras.losses.sparse_categorical_crossentropy(lbl, preds)
                grad = tape.gradient(loss, adv)
                adv = adv + alpha * tf.sign(grad)
                adv = tf.clip_by_value(adv, orig - epsilon, orig + epsilon)
                adv = tf.clip_by_value(adv, 0.0, 255.0)
            out[start:end] = adv.numpy()
        return out

    for eps in [2, 4, 8]:
        X_adv = pgd_attack_in_batches(full_model, X_test, y_test, epsilon=eps)
        y_prob = predict_in_batches(full_model, X_adv)
        y_pred = np.argmax(y_prob, axis=1)
        acc = accuracy_score(y_test, y_pred)
        adv_rows.append({"Attack": "PGD (10-step)", "Epsilon": eps, "Accuracy": acc,
                          "Accuracy Drop (pp)": (acc_clean - acc) * 100})
        print(f"PGD  epsilon={eps:>3d}/255 -> Acc={acc:.4f}  (drop={100*(acc_clean-acc):.2f} pp)")

adv_df = pd.DataFrame(adv_rows)
adv_df.to_csv(os.path.join(OUTPUT_DIR, "adversarial_robustness_results.csv"), index=False)

plt.figure(figsize=(7, 5))
for attack_name, sub in adv_df.groupby("Attack"):
    plt.plot(sub["Epsilon"], sub["Accuracy"], marker="o", label=attack_name)
plt.axhline(acc_clean, color="gray", linestyle="--", label="Clean accuracy")
plt.xlabel("Perturbation budget (epsilon, 0-255 pixel scale)")
plt.ylabel("Accuracy")
plt.title("Adversarial Robustness")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "adversarial_robustness_curve.png"), dpi=600)
plt.show()

print("\nAdversarial robustness summary:")
print(adv_df.to_string(index=False))

# ==================================================================================
# PART F: CROSS-DOMAIN / CROSS-SEASON GENERALIZATION TEST
# ==================================================================================
print("\n" + "=" * 78)
print("PART F: CROSS-DOMAIN / CROSS-SEASON GENERALIZATION TEST")
print("=" * 78)

if CROSS_DOMAIN_PATH is None or not os.path.isdir(CROSS_DOMAIN_PATH):
    print(
        "CROSS_DOMAIN_PATH is not set (or the folder doesn't exist) -> skipping PART F.\n"
        "To run this test: collect a small new batch of rose-leaf images from a "
        "different season / lighting / camera, organize them the same way as "
        "COMBINED_PATH (one sub-folder per class name, matching class_names below), "
        "set CROSS_DOMAIN_PATH to that folder, and re-run this script.\n"
        f"Expected class folders: {class_names}"
    )
else:
    cd_paths, cd_labels = [], []
    for cls in class_names:
        folder = os.path.join(CROSS_DOMAIN_PATH, cls)
        if not os.path.isdir(folder):
            print(f"  WARNING: no folder found for class '{cls}' under {CROSS_DOMAIN_PATH} — skipping this class.")
            continue
        for img in os.listdir(folder):
            cd_paths.append(os.path.join(folder, img))
            cd_labels.append(class_to_idx[cls])

    X_cross = np.stack([load_image_array(p) for p in cd_paths]).astype(np.float32)
    y_cross = np.array(cd_labels)

    y_prob_cross = predict_in_batches(full_model, X_cross)
    y_pred_cross = np.argmax(y_prob_cross, axis=1)
    acc_cross = accuracy_score(y_cross, y_pred_cross)
    _, _, f1_cross, _ = precision_recall_fscore_support(y_cross, y_pred_cross, average="weighted", zero_division=0)

    cross_domain_result = pd.DataFrame([{
        "Test set": "Original held-out test (same domain)",
        "N": len(y_test), "Accuracy": acc_clean, "F1": f1_clean
    }, {
        "Test set": "Cross-domain / cross-season set",
        "N": len(y_cross), "Accuracy": acc_cross, "F1": f1_cross
    }])
    cross_domain_result["Accuracy Drop (pp) vs same-domain"] = \
        [0.0, (acc_clean - acc_cross) * 100]
    cross_domain_result.to_csv(os.path.join(OUTPUT_DIR, "cross_domain_results.csv"), index=False)
    print(cross_domain_result.to_string(index=False))

# ==================================================================================
# PART G: LINEAR PROBING (frozen encoder + a single linear layer)
# ==================================================================================
print("\n" + "=" * 78)
print("PART G: LINEAR PROBING — isolating what the SSL features alone can do")
print("=" * 78)

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = tf.cast(img, tf.float32)
    img.set_shape([IMG_SIZE, IMG_SIZE, 3])
    return img, label

def make_dataset(paths, labs, shuffle=False, seed=None):
    ds = tf.data.Dataset.from_tensor_slices((paths, labs))
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(2000, seed=seed)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(AUTOTUNE)
    return ds

# small internal train/val split of the training pool (val only used for early stopping)
lp_tr_paths, lp_va_paths, lp_tr_labels, lp_va_labels = train_test_split(
    trainpool_paths, trainpool_labels, test_size=0.11, random_state=FIXED_TEST_SEED,
    stratify=trainpool_labels
)
lp_train_ds = make_dataset(lp_tr_paths, lp_tr_labels, shuffle=True, seed=FIXED_TEST_SEED)
lp_val_ds = make_dataset(lp_va_paths, lp_va_labels, shuffle=False)
test_ds_for_probe = make_dataset(test_paths, test_labels, shuffle=False)

def build_linear_probe(use_ssl_encoder, seed=FIXED_TEST_SEED):
    tf.keras.utils.set_random_seed(seed)
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
    if use_ssl_encoder:
        backbone = tf.keras.models.load_model(SSL_ENCODER_PATH)
    else:
        backbone = tf.keras.applications.MobileNetV2(
            include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3)
        )
    backbone.trainable = False   # FROZEN: this is what makes it a "linear probe"
    x = backbone(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)   # ONE linear layer, nothing else
    model = Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

def train_and_eval_probe(name, use_ssl_encoder, epochs=25):
    model = build_linear_probe(use_ssl_encoder=use_ssl_encoder)
    callbacks = [tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True)]
    model.fit(lp_train_ds, validation_data=lp_val_ds, epochs=epochs, callbacks=callbacks, verbose=0)
    y_prob = model.predict(test_ds_for_probe, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)
    acc = accuracy_score(test_labels, y_pred)
    _, _, f1, _ = precision_recall_fscore_support(test_labels, y_pred, average="weighted", zero_division=0)
    print(f"{name:28s} -> linear-probe test accuracy = {acc:.4f}, F1 = {f1:.4f}")
    return {"Representation": name, "Linear-Probe Accuracy": acc, "Linear-Probe F1": f1}

probe_rows = []
probe_rows.append(train_and_eval_probe("ImageNet-only (no SSL)", use_ssl_encoder=False))
probe_rows.append(train_and_eval_probe("SimCLR-pretrained (SSL)", use_ssl_encoder=True))
probe_rows.append({
    "Representation": "Full fine-tuned model (SimCLR+CBAM+head)",
    "Linear-Probe Accuracy": acc_clean, "Linear-Probe F1": f1_clean
})  # reference row — NOT a linear probe, shown for comparison only

probe_df = pd.DataFrame(probe_rows)
probe_df.to_csv(os.path.join(OUTPUT_DIR, "linear_probing_results.csv"), index=False)

print("\nLinear probing comparison (isolates the value of SSL pretraining alone):")
print(probe_df.to_string(index=False))
print(
    "\nInterpretation: the gap between 'ImageNet-only' and 'SimCLR-pretrained' linear-probe "
    "accuracy is the discriminative signal SimCLR pretraining adds BEFORE any fine-tuning, "
    "CBAM, or classification head are introduced. The remaining gap up to the full fine-tuned "
    "model's accuracy is what fine-tuning + CBAM + the classification head contribute on top."
)

plt.figure(figsize=(7, 4))
plt.bar(probe_df["Representation"], probe_df["Linear-Probe Accuracy"], color=["#C44E52", "#55A868", "#4C72B0"])
plt.ylabel("Test Accuracy")
plt.title("Linear Probing: SSL Feature Quality vs Full Fine-Tuned Model")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "linear_probing_comparison.png"), dpi=600)
plt.show()

# ==================================================================================
# PART H: EMBEDDING-QUALITY METRICS (Silhouette score, Davies-Bouldin index)
# ==================================================================================
print("\n" + "=" * 78)
print("PART H: EMBEDDING-QUALITY METRICS")
print("=" * 78)

# 512-d embedding just before BatchNorm/Dropout/final Dense in the full model
feature_model = tf.keras.Model(inputs=full_model.input, outputs=full_model.layers[-4].output)
embeddings_512d = feature_model.predict(test_ds_for_probe, verbose=0)
y_true_embed = test_labels

tsne = TSNE(n_components=2, perplexity=30, random_state=FIXED_TEST_SEED, init="pca")
embeddings_2d = tsne.fit_transform(embeddings_512d)

def embedding_quality(features, labels_arr, name):
    sil = silhouette_score(features, labels_arr)
    dbi = davies_bouldin_score(features, labels_arr)
    return {"Representation": name, "Dim": features.shape[1],
            "Silhouette Score (higher=better)": sil,
            "Davies-Bouldin Index (lower=better)": dbi}

embed_rows = [
    embedding_quality(embeddings_512d, y_true_embed, "512-d embedding (raw, full model)"),
    embedding_quality(embeddings_2d, y_true_embed, "2-D t-SNE projection (full model)"),
]

# Optional: also quantify SSL-only (no CBAM/fine-tuning) embedding quality for comparison
ssl_only_backbone = tf.keras.models.load_model(SSL_ENCODER_PATH)
ssl_inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
ssl_x = tf.keras.applications.mobilenet_v2.preprocess_input(ssl_inputs)
ssl_x = ssl_only_backbone(ssl_x, training=False)
ssl_x = layers.GlobalAveragePooling2D()(ssl_x)
ssl_feature_model = tf.keras.Model(ssl_inputs, ssl_x)
ssl_embeddings = ssl_feature_model.predict(test_ds_for_probe, verbose=0)
embed_rows.append(embedding_quality(ssl_embeddings, y_true_embed, "SSL-only pooled embedding (no CBAM/head)"))

embed_df = pd.DataFrame(embed_rows)
embed_df.to_csv(os.path.join(OUTPUT_DIR, "embedding_quality_metrics.csv"), index=False)

print(embed_df.to_string(index=False))
print(
    "\nInterpretation: Silhouette closer to +1 and Davies-Bouldin closer to 0 both mean "
    "tighter, better-separated class clusters. Comparing 'SSL-only' vs the full model's "
    "'512-d embedding' quantifies, with numbers rather than just a t-SNE picture, how much "
    "CBAM + fine-tuning further sharpen class separability on top of the SSL features."
)

plt.figure(figsize=(9, 7))
colors = plt.cm.tab10(np.linspace(0, 1, NUM_CLASSES))
for i, cls in enumerate(class_names):
    idx = np.where(y_true_embed == i)[0]
    plt.scatter(embeddings_2d[idx, 0], embeddings_2d[idx, 1], label=cls, alpha=0.7, color=colors[i])
sil_2d = embed_df.loc[embed_df["Representation"].str.contains("t-SNE"), "Silhouette Score (higher=better)"].values[0]
plt.title(f"t-SNE of Test-Set Embeddings (Silhouette = {sil_2d:.3f})")
plt.xlabel("t-SNE Component 1")
plt.ylabel("t-SNE Component 2")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "tsne_with_quality_score.png"), dpi=600)
plt.show()

# ==================================================================================
# SUMMARY OF SAVED FILES
# ==================================================================================
print("\n" + "=" * 78)
print(f"All outputs saved under: {OUTPUT_DIR}/")
print("=" * 78)
for f in sorted(os.listdir(OUTPUT_DIR)):
    if f.endswith((".csv", ".png")):
        print(" -", f)